# Pristine silicon at glancing incidence

This notebook shows a convergent electron probe moving across a pristine,
unstrained silicon slab. The probe is formed by Fourier transforming a circular
aperture, so its focused intensity is an Airy pattern. The interactive viewer
controls the convergence semi-angle, defocus, and surface landing position.


In [17]:
%matplotlib widget

from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", ".2")

# This viewer caches long side views; CPU execution avoids competing with
# reconstruction notebooks for GPU memory. Restart the kernel before running.
os.environ["JAX_PLATFORMS"] = "cpu"

repo_root = Path.cwd()
if not (repo_root / "wide_angle_propagation").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import abtem
from ase import Atoms
from ase.build import bulk
import ipywidgets as widgets
import jax
import jax.numpy as jnp
from IPython.display import display
from matplotlib.colors import ListedColormap, PowerNorm
import matplotlib.pyplot as plt
import numpy as np

from wide_angle_propagation.propagation_methods import (
    build_atom_aligned_screen_partition_1d,
    energy2wavelength,
    project_potential_to_screens_1d,
    simulate_projected_as_screens_1d,
)
from wide_angle_propagation.ptychography_1d import (
    GlancingSideviewCache1D,
)

abtem.config.set({"device": "cpu", "precision": "float64"})
jax.config.update("jax_enable_x64", True)


## Build one pristine silicon slab

A finite diamond-silicon slab is built from explicit atoms and rendered once with the Lobato potential. It is periodic only through the out-of-plane projection width; the surface-normal and propagation directions are finite, so no unit-cell image is cut or stitched at the surface.

In [18]:
energy_eV = 30_000.0
glancing_angle_deg = 2.0
beam_tilt_rad = -np.deg2rad(glancing_angle_deg)
convergence_semiangle_mrad = 15.0
defocus_A = 0.0
propagation_length_A = 1000.0
slab_depth_A = 50.0
vacuum_above_A = 40.0
vacuum_below_A = 20.0
si_lattice_A = 5.431

si_unit = bulk("Si", "diamond", a=si_lattice_A, cubic=True)
minimum_projected_si_spacing_A = si_lattice_A / 4.0
surface_clearance_A = minimum_projected_si_spacing_A / 2.0
surface_global_A = vacuum_below_A + slab_depth_A
transverse_extent_A = vacuum_below_A + slab_depth_A + vacuum_above_A

# Build the finite slab from explicit atoms. The repeated bulk coordinates
# point into the material and are then placed below the surface.
n_depth_cells = int(np.ceil(slab_depth_A / si_lattice_A)) + 1
n_length_cells = int(np.ceil(propagation_length_A / si_lattice_A)) + 1
repeated_si = si_unit.repeat((n_depth_cells, 1, n_length_cells))
slab_positions_A = repeated_si.positions
inside_slab = (
    (slab_positions_A[:, 0] <= slab_depth_A - surface_clearance_A + 1e-9)
    & (slab_positions_A[:, 2] < propagation_length_A - 1e-9)
)
slab_positions_A = slab_positions_A[inside_slab].copy()
slab_positions_A[:, 0] = surface_global_A - surface_clearance_A - slab_positions_A[:, 0]
si_slab = Atoms(
    symbols=["Si"] * len(slab_positions_A),
    positions=slab_positions_A,
    cell=(transverse_extent_A, si_lattice_A, propagation_length_A),
    pbc=(False, True, False),
)

builder = abtem.Potential(
    si_slab,
    sampling=(0.115, 0.20),
    slice_thickness=float(si_unit.cell.lengths()[1]),
    projection="finite",
    parametrization="lobato",
    plane="xz",
    periodic=False,
    device="cpu",
)
finite_projected = np.asarray(builder.build(lazy=False).array)[0]
du, ds = (float(value) for value in builder.sampling)
n_u, n_s = finite_projected.shape
s_A = np.arange(n_s) * ds
u_A = np.arange(n_u) * du - surface_global_A
silicon_potential = finite_projected.T / float(si_unit.cell.lengths()[1])
axial_atom_planes_A = np.unique(np.round(slab_positions_A[:, 2], decimals=8))
screen_partition = build_atom_aligned_screen_partition_1d(
    s_A,
    axial_atom_planes_A,
    sub_screens_per_plane=3,
    reference_potential=silicon_potential,
    screen_offset_scale=0.7,
)
projected_screen_potential = project_potential_to_screens_1d(
    jnp.asarray(silicon_potential), screen_partition
)

wavelength_A = float(energy2wavelength(energy_eV))
tilted_wave_period_A = wavelength_A / abs(np.sin(beam_tilt_rad))
focused_airy_first_zero_A = (
    0.61 * wavelength_A / (convergence_semiangle_mrad * 1e-3)
)
sampling_diagnostics = {
    "samples across focused Airy radius (u)": focused_airy_first_zero_A / du,
    "samples per tilted-wave period (u)": tilted_wave_period_A / du,
    "samples per minimum Si spacing (u)": minimum_projected_si_spacing_A / du,
    "samples per minimum Si spacing (s)": minimum_projected_si_spacing_A / ds,
    "transverse Nyquist angle (degrees)": np.rad2deg(np.arcsin(min(1.0, wavelength_A / (2.0 * du)))),
}
assert sampling_diagnostics["samples across focused Airy radius (u)"] >= 12.0
assert sampling_diagnostics["samples per tilted-wave period (u)"] >= 12.0
assert sampling_diagnostics["samples per minimum Si spacing (u)"] >= 10.0
assert sampling_diagnostics["samples per minimum Si spacing (s)"] >= 6.0

print({
    "fine source potential shape (not propagated)": silicon_potential.shape,
    "propagated phase-screen shape": projected_screen_potential.shape,
    "sampling (ds, du) Å": (ds, du),
    "propagation length Å": float(s_A[-1]),
    "silicon depth Å": slab_depth_A,
    "explicit Si atoms": len(si_slab),
    "AS propagation steps": screen_partition.n_screens,
    "fine-grid / AS-step ratio": n_s / screen_partition.n_screens,
    "outermost atom depth Å": -surface_clearance_A,
    "minimum / maximum potential": (float(silicon_potential.min()), float(silicon_potential.max())),
    "sampling checks": sampling_diagnostics,
})


{'fine source potential shape (not propagated)': (5000, 957), 'propagated phase-screen shape': (2211, 957), 'sampling (ds, du) Å': (0.2, 0.11494252873563218), 'propagation length Å': 999.8000000000001, 'silicon depth Å': 50.0, 'explicit Si atoms': 13635, 'AS propagation steps': 2211, 'fine-grid / AS-step ratio': 2.261420171867933, 'outermost atom depth Å': -0.678875, 'minimum / maximum potential': (0.0, 119.19617362905802), 'sampling checks': {'samples across focused Airy radius (u)': 24.691990505907235, 'samples per tilted-wave period (u)': 17.397961339770358, 'samples per minimum Si spacing (u)': 11.812425000000001, 'samples per minimum Si spacing (s)': 6.788749999999999, 'transverse Nyquist angle (degrees)': 17.673357866679364}}


## Form the aperture-limited probe and propagate it

The complex pupil is a hard circular aperture with radius
$q_{max}=\alpha/\lambda$. A quadratic pupil phase
$\exp[-i\pi\lambda\Delta f(q_x^2+q_y^2)]$ applies defocus before a 2D
inverse FFT forms the probe. The 1D multislice model uses the central line of
that genuine 2D Airy probe; the viewer shows the full 2D intensity.

Propagation uses exact angular-spectrum steps between atom-aligned phase
screens. Each atomic-plane Voronoi cell is integrated into three local
screens: one remains on the plane and two describe its axial tails. Tail
centroids are contracted towards the plane by a calibrated factor of 0.7.
The fine 0.2 Angstrom potential remains the source of every integrated phase,
so no potential samples are discarded. The propagation itself always uses
the faster 2211-screen atom-aligned representation.

Changing convergence or defocus requires rebuilding the cached propagation, so
those controls use an **Apply beam settings** button. The landing-position
slider remains instantaneous once the cache is built.


In [19]:
scan_coordinates_A = np.linspace(400.0, 600.0, 21)
beam_centres_at_entrance_A = -scan_coordinates_A * np.tan(beam_tilt_rad)
beam_coordinates_A = (np.arange(n_u) - n_u // 2) * du
aperture_frequencies_inv_A = np.fft.fftfreq(n_u, d=du)


def make_aperture_limited_probe(convergence_mrad, defocus_A):
    """Return a 1D probe line, full 2D probe, and circular pupil.

    Defocus follows the pupil-phase convention
    exp(-i pi lambda defocus q^2). The returned fields have unit peak
    amplitude, matching the former Gaussian probe's normalization.
    """
    convergence_rad = float(convergence_mrad) * 1e-3
    aperture_radius_inv_A = np.sin(convergence_rad) / wavelength_A
    qx, qy = np.meshgrid(
        aperture_frequencies_inv_A,
        aperture_frequencies_inv_A,
        indexing="xy",
    )
    q_squared = qx**2 + qy**2
    pupil_mask = q_squared <= aperture_radius_inv_A**2
    pupil = pupil_mask * np.exp(-1j * np.pi * wavelength_A * float(defocus_A) * q_squared)
    probe_2d = np.fft.fftshift(np.fft.ifft2(pupil))
    probe_2d /= np.max(np.abs(probe_2d)) + 1e-30
    probe_line = probe_2d[n_u // 2].copy()
    probe_line /= np.max(np.abs(probe_line)) + 1e-30
    return probe_line, probe_2d, pupil_mask


def make_scanned_probes(probe_line):
    """Shift the Airy envelope to each entrance centre and add beam tilt."""
    probe_spectrum = np.fft.fft(probe_line)
    carrier = np.exp(
        2j * np.pi * np.sin(beam_tilt_rad) * u_A / wavelength_A
    )
    array_centre_A = float(u_A[n_u // 2])
    probes = []
    for center_A in beam_centres_at_entrance_A:
        shift_A = float(center_A) - array_centre_A
        shifted_envelope = np.fft.ifft(
            probe_spectrum
            * np.exp(-2j * np.pi * aperture_frequencies_inv_A * shift_A)
        )
        probes.append(shifted_envelope * carrier)
    return jnp.asarray(np.stack(probes), dtype=jnp.complex128)


detector_frequencies = np.fft.fftshift(np.fft.fftfreq(n_u, d=du))
detector_angles_mrad = 1e3 * np.arcsin(
    np.clip(wavelength_A * detector_frequencies, -1.0, 1.0)
)
def block_average_2d(array, stride_s, stride_u):
    usable_s = (array.shape[0] // stride_s) * stride_s
    usable_u = (array.shape[1] // stride_u) * stride_u
    trimmed = array[:usable_s, :usable_u]
    return trimmed.reshape(
        usable_s // stride_s, stride_s, usable_u // stride_u, stride_u
    ).mean(axis=(1, 3))


def block_average_1d(array, stride):
    usable = (array.shape[0] // stride) * stride
    return array[:usable].reshape(usable // stride, stride).mean(axis=1)


@jax.jit
def propagate_screened_probe(initial_wave):
    return simulate_projected_as_screens_1d(
        initial_wave,
        projected_screen_potential,
        screen_partition.screen_positions,
        du,
        energy_eV,
        domain_start=screen_partition.domain_start,
        domain_end=screen_partition.domain_end,
        return_diagnostics=True,
    )


def build_sideview_cache(convergence_mrad, defocus_value_A):
    probe_line, probe_2d, pupil_mask = make_aperture_limited_probe(
        convergence_mrad, defocus_value_A
    )
    input_probes = make_scanned_probes(probe_line)
    axial_stride, transverse_stride = 2, 2
    fields, intensities = [], []
    exit_waves, detector_waves, detector_intensities = [], [], []
    wavefront_coordinates = None
    for initial_wave in input_probes:
        exit_wave, _, diagnostics = propagate_screened_probe(initial_wave)
        wavefields = diagnostics["wavefronts"]
        wavefront_coordinates = diagnostics["wavefront_coordinates"]
        detector_wave = jnp.fft.fftshift(jnp.fft.fft(exit_wave))
        fields.append(
            block_average_2d(wavefields, axial_stride, transverse_stride)
        )
        intensities.append(
            block_average_2d(
                jnp.abs(wavefields) ** 2, axial_stride, transverse_stride
            )
        )
        exit_waves.append(exit_wave)
        detector_waves.append(detector_wave)
        detector_intensities.append(jnp.abs(detector_wave) ** 2)
    detector_intensities_full = jnp.stack(detector_intensities)
    metadata = {
        "specimen": "pristine silicon",
        "glancing_angle_deg": glancing_angle_deg,
        "probe": "2D circular-aperture FFT, central 1D line",
        "convergence_semiangle_mrad": float(convergence_mrad),
        "defocus_A": float(defocus_value_A),
        "propagator": "atom-aligned exact angular spectrum",
        "subscreens_per_atom_plane": 3,
        "screen_offset_scale": 0.7,
        "phase_screen_count": screen_partition.n_screens,
        "fine_slice_count": n_s,
    }
    return (
        GlancingSideviewCache1D(
            scan_indices=jnp.arange(len(scan_coordinates_A)),
            window_starts=jnp.zeros(len(scan_coordinates_A), dtype=jnp.int32),
            scan_coordinates=jnp.asarray(scan_coordinates_A),
            local_s_coordinates=block_average_1d(
                wavefront_coordinates, axial_stride
            ),
            sideview_u_coordinates=block_average_1d(
                jnp.asarray(u_A), transverse_stride
            ),
            transverse_coordinates=jnp.asarray(u_A),
            sideview_wavefields=jnp.stack(fields).astype(jnp.complex64),
            sideview_intensities=jnp.stack(intensities).astype(jnp.float32),
            exit_waves=jnp.stack(exit_waves).astype(jnp.complex64),
            detector_waves=jnp.stack(detector_waves).astype(jnp.complex64),
            detector_intensities=detector_intensities_full.astype(jnp.float32),
            metadata=metadata,
        ),
        probe_line, probe_2d, pupil_mask,
    )


sideviews, input_probe_line, input_probe_2d, aperture_mask = build_sideview_cache(
    convergence_semiangle_mrad, defocus_A
)
print(
    f"Cached {len(scan_coordinates_A)} atom-aligned side views with "
    f"{screen_partition.n_screens} screens; "
    f"alpha = {convergence_semiangle_mrad:.1f} mrad, "
    f"defocus = {defocus_A:.1f} Å."
)


Cached 21 atom-aligned side views with 2211 screens; alpha = 15.0 mrad, defocus = 0.0 Å.


## Interactive side viewer

Set the convergence semi-angle and defocus, then click **Apply beam settings**
to recalculate the propagation cache. The beam panel displays the actual 2D
FFT intensity at the input plane. Move the landing slider to inspect any cached
scan without rerunning the simulation. The display floor only changes the visual contrast: it does not change the simulated wave or intensity.


In [20]:
side_s_A = np.asarray(sideviews.local_s_coordinates)
side_u_A = np.asarray(sideviews.sideview_u_coordinates)
potential_extent = [s_A[0] - ds / 2.0, s_A[-1] + ds / 2.0, u_A[0] - du / 2.0, u_A[-1] + du / 2.0]
side_ds_A = float(side_s_A[1] - side_s_A[0])
side_du_A = float(side_u_A[1] - side_u_A[0])
sideview_extent = [side_s_A[0] - side_ds_A / 2.0, side_s_A[-1] + side_ds_A / 2.0, side_u_A[0] - side_du_A / 2.0, side_u_A[-1] + side_du_A / 2.0]
positive_exit = (u_A >= 0.0) & (u_A <= 35.0)
potential_max = float(np.max(silicon_potential))
# A low floor reveals weak reflected intensity while the steep alpha ramp
# keeps it visually distinct from the high-flux incident core.
beam_display_floor = 1e-5
beam_image_half_width_A = 18.0


def display_beam_intensity(normalized_intensity, floor):
    """Map relative intensity to display values without boosting weak tails."""
    clipped = np.clip(normalized_intensity, floor, 1.0)
    return np.clip(
        (np.log10(clipped) - np.log10(floor)) / -np.log10(floor),
        0.0,
        1.0,
    )

with plt.ioff():
    figure = plt.figure(figsize=(20, 9.5), constrained_layout=True)


def draw_scan(scan_index):
    scan_index = int(scan_index)
    figure.clear()
    grid = figure.add_gridspec(
        3, 3, height_ratios=(1.7, 2.6, 2.2), width_ratios=(1.0, 1.0, 1.0)
    )
    side_ax = figure.add_subplot(grid[0, :])
    beam_ax = figure.add_subplot(grid[1, 1])
    exit_ax = figure.add_subplot(grid[2, 0])
    phase_ax = figure.add_subplot(grid[2, 1])
    detector_ax = figure.add_subplot(grid[2, 2])

    side_ax.set_facecolor("#10151c")
    side_ax.axhspan(
        -slab_depth_A, 0.0, color="#303946", alpha=1.0,
        zorder=-1, label="Si slab"
    )
    # Keep weak potential values transparent while rendering the atomic
    # structure strongly against an opaque dark specimen background.
    potential_colours = plt.get_cmap("magma")(np.linspace(0.16, 1.0, 256))
    potential_colours[:, 3] = np.linspace(0.0, 1.0, 256) ** 2
    potential_cmap = ListedColormap(potential_colours)
    potential_cmap.set_under(alpha=0.0)
    side_ax.imshow(
        silicon_potential.T,
        origin="lower",
        aspect="equal",
        extent=potential_extent,
        cmap=potential_cmap,
        norm=PowerNorm(gamma=0.45, vmin=max(potential_max * 1e-4, 1e-12), vmax=potential_max),
        alpha=0.88,
        interpolation="hanning",
    )
    scan_intensity = np.asarray(sideviews.sideview_intensities[scan_index]).T
    normalized_intensity = scan_intensity / (float(scan_intensity.max()) + 1e-30)
    beam_display = display_beam_intensity(
        normalized_intensity, beam_display_floor
    )
    # Transparency is tied to actual relative intensity. This prevents the
    # log colour transform from making weak Airy tails look beam-strength.
    beam_alpha = 0.98 * beam_display**1.45
    intensity_image = side_ax.imshow(
        beam_display,
        origin="lower",
        aspect="equal",
        extent=sideview_extent,
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        alpha=beam_alpha,
        interpolation="hanning",
    )
    landing_A = float(scan_coordinates_A[scan_index])
    centreline_A = np.tan(beam_tilt_rad) * (side_s_A - landing_A)
    side_ax.plot(side_s_A, centreline_A, "w--", linewidth=1.2, label="nominal beam centre")
    side_ax.scatter([landing_A], [0.0], color="cyan", edgecolor="black", s=45, zorder=5, label="surface landing")
    side_ax.set(
        xlim=(s_A[0], s_A[-1]), ylim=(-55.0, 35.0),
        xlabel="s (Å)", ylabel="u (Å)",
        title=f"Pristine Si; landing at s = {landing_A:.1f} Å",
        aspect="equal",
    )
    side_ax.legend(loc="lower left")

    probe_intensity = np.abs(input_probe_2d) ** 2
    probe_intensity /= probe_intensity.max() + 1e-30
    beam_image = beam_ax.imshow(
        display_beam_intensity(probe_intensity, beam_display_floor),
        origin="lower",
        extent=[beam_coordinates_A[0], beam_coordinates_A[-1], beam_coordinates_A[0], beam_coordinates_A[-1]],
        cmap="inferno",
        vmin=0.0,
        vmax=1.0,
        interpolation="nearest",
    )
    beam_ax.set(
        xlim=(-beam_image_half_width_A, beam_image_half_width_A),
        ylim=(-beam_image_half_width_A, beam_image_half_width_A),
        xlabel="x (Å)",
        ylabel="y (Å)",
        title=(
            f"Input beam: α = {convergence_semiangle_mrad:.1f} mrad, "
            f"Δf = {defocus_A:.0f} Å"
        ),
        aspect="equal",
    )

    exit_wave = np.asarray(sideviews.exit_waves[scan_index])
    positive_exit_wave = exit_wave[positive_exit]
    positive_exit_u_A = u_A[positive_exit]
    positive_exit_intensity = np.abs(positive_exit_wave) ** 2
    exit_ax.plot(positive_exit_u_A, positive_exit_intensity)
    exit_ax.set(xlim=(0.0, 35.0), ylim=(0.0, max(float(positive_exit_intensity.max()) * 1.03, 1e-30)), xlabel="u (Å)", ylabel="intensity", title="Positive-u exit intensity")

    phase_is_valid = positive_exit_intensity > max(float(positive_exit_intensity.max()) * 1e-5, 1e-30)
    phase_u_A = positive_exit_u_A[phase_is_valid]
    unwrapped_phase = np.unwrap(np.angle(positive_exit_wave[phase_is_valid]))
    phase_ax.plot(phase_u_A, unwrapped_phase, linewidth=1.0)
    phase_ax.set(xlim=(0.0, 35.0), xlabel="u (Å)", ylabel="phase (rad)", title="Unwrapped exit-wave phase")

    positive_angles = detector_angles_mrad > 0.0
    positive_detector_intensity = np.asarray(sideviews.detector_intensities[scan_index])[positive_angles]
    detector_ax.plot(detector_angles_mrad[positive_angles], positive_detector_intensity, linewidth=1.2)
    detector_ax.axvline(-1e3 * beam_tilt_rad, color="tab:red", linestyle="--", linewidth=0.9, label="specular")
    detector_ax.set(xlim=(0.0, 80.0), xlabel="positive detector angle (mrad)", ylabel="intensity", title="Positive-angle far field (linear scale)")
    detector_ax.set_ylim(bottom=0.0)
    detector_ax.legend(fontsize=8, frameon=False)
    figure.canvas.draw_idle()


landing_slider = widgets.SelectionSlider(
    options=[(f"{index:03d} — {landing:6.1f} Å", index) for index, landing in enumerate(scan_coordinates_A)],
    value=len(scan_coordinates_A) // 2,
    description="landing position",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="720px"),
)
convergence_slider = widgets.FloatSlider(
    value=convergence_semiangle_mrad,
    min=5.0,
    max=30.0,
    step=1.0,
    description="convergence α (mrad)",
    continuous_update=False,
    readout_format=".1f",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="390px"),
)
display_floor_slider = widgets.FloatLogSlider(
    value=beam_display_floor,
    base=10,
    min=-6.0,
    max=-2.0,
    step=0.25,
    description="reflection display floor",
    continuous_update=False,
    readout_format=".1e",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)
defocus_slider = widgets.FloatSlider(
    value=defocus_A,
    min=-500.0,
    max=500.0,
    step=10.0,
    description="defocus Δf (Å)",
    continuous_update=False,
    readout_format=".0f",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="390px"),
)
apply_beam_button = widgets.Button(
    description="Apply beam settings",
    button_style="primary",
    icon="refresh",
    layout=widgets.Layout(width="190px"),
)
beam_status = widgets.HTML(
    value=(
        f"<span style='color:#2b7'>Cached α={convergence_semiangle_mrad:.1f} mrad, "
        f"Δf={defocus_A:.0f} Å</span>"
    )
)


def apply_beam_settings(_):
    global sideviews, input_probe_line, input_probe_2d, aperture_mask
    global convergence_semiangle_mrad, defocus_A
    apply_beam_button.disabled = True
    beam_status.value = "<b>Recalculating the propagation cache…</b>"
    try:
        new_convergence = float(convergence_slider.value)
        new_defocus = float(defocus_slider.value)
        new_cache, new_line, new_2d, new_mask = build_sideview_cache(
            new_convergence, new_defocus
        )
        sideviews = new_cache
        input_probe_line = new_line
        input_probe_2d = new_2d
        aperture_mask = new_mask
        convergence_semiangle_mrad = new_convergence
        defocus_A = new_defocus
        beam_status.value = (
            f"<span style='color:#2b7'>Cached α={new_convergence:.1f} mrad, "
            f"Δf={new_defocus:.0f} Å</span>"
        )
        draw_scan(landing_slider.value)
    except Exception as error:
        beam_status.value = f"<span style='color:#b22'><b>Beam update failed:</b> {error}</span>"
        raise
    finally:
        apply_beam_button.disabled = False


def update_display_floor(change):
    global beam_display_floor
    beam_display_floor = float(change["new"])
    draw_scan(landing_slider.value)


landing_slider.observe(lambda change: draw_scan(change["new"]), names="value")
display_floor_slider.observe(update_display_floor, names="value")
apply_beam_button.on_click(apply_beam_settings)
draw_scan(landing_slider.value)
beam_controls = widgets.HBox([
    convergence_slider, defocus_slider, apply_beam_button
])
display_controls = widgets.HBox([display_floor_slider])
display(widgets.VBox([
    beam_controls, display_controls, beam_status, landing_slider, figure.canvas
]))
